In [1]:
import io
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import CubicSpline
from datetime import datetime, timedelta

In [2]:
CSV_SOURCE = """Dates,Prices
10/31/20,1.01E+01
11/30/20,1.03E+01
12/31/20,1.10E+01
1/31/21,1.09E+01
2/28/21,1.09E+01
3/31/21,1.09E+01
4/30/21,1.04E+01
5/31/21,9.84E+00
6/30/21,1.00E+01
7/31/21,1.01E+01
8/31/21,1.03E+01
9/30/21,1.02E+01
10/31/21,1.01E+01
11/30/21,1.12E+01
12/31/21,1.14E+01
1/31/22,1.15E+01
2/28/22,1.18E+01
3/31/22,1.15E+01
4/30/22,1.07E+01
5/31/22,1.07E+01
6/30/22,1.04E+01
7/31/22,1.05E+01
8/31/22,1.04E+01
9/30/22,1.08E+01
10/31/22,1.10E+01
11/30/22,1.16E+01
12/31/22,1.16E+01
1/31/23,1.21E+01
2/28/23,1.17E+01
3/31/23,1.20E+01
4/30/23,1.15E+01
5/31/23,1.12E+01
6/30/23,1.09E+01
7/31/23,1.14E+01
8/31/23,1.11E+01
9/30/23,1.15E+01
10/31/23,1.18E+01
11/30/23,1.22E+01
12/31/23,1.28E+01
1/31/24,1.26E+01
2/29/24,1.24E+01
3/31/24,1.27E+01
4/30/24,1.21E+01
5/31/24,1.14E+01
6/30/24,1.15E+01
7/31/24,1.16E+01
8/31/24,1.15E+01
9/30/24,1.18E+01
"""

def load_monthly_prices(csv_source: str) -> pd.DataFrame:
    df = pd.read_csv(io.StringIO(csv_source))
    # Parse dates (US-style month/day/year) assuming 20xx (CSV provides 2-digit years)
    df['Date'] = pd.to_datetime(df['Dates'])
    # Convert scientific notation strings to floats
    df['Price'] = df['Prices'].astype(float)
    df = df[['Date', 'Price']].sort_values('Date').reset_index(drop=True)
    return df

monthly_df = load_monthly_prices(CSV_SOURCE)

C:\Users\Public\Documents\Wondershare\CreatorTemp\ipykernel_12292\925109924.py:55: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Date'] = pd.to_datetime(df['Dates'])


In [9]:
from datetime import datetime
import pandas as pd

def price_storage_contract(
    injection_dates, withdrawal_dates,
    injection_volumes, withdrawal_volumes,
    injection_rate, withdrawal_rate,
    max_storage_capacity,
    storage_fee_per_month,
    transport_fee_per_move,
    estimate_price_func
):
    """
    Prototype pricing model for natural gas storage contract.
    
    Parameters:
    - injection_dates: list of str 'YYYY-MM-DD'
    - withdrawal_dates: list of str 'YYYY-MM-DD'
    - injection_volumes: list of floats (MMBtu)
    - withdrawal_volumes: list of floats (MMBtu)
    - injection_rate: $ cost per MMBtu injected
    - withdrawal_rate: $ cost per MMBtu withdrawn
    - max_storage_capacity: maximum MMBtu that can be stored
    - storage_fee_per_month: fixed monthly storage fee ($)
    - transport_fee_per_move: fixed fee per injection/withdrawal ($)
    - estimate_price_func: function(date) -> price
    
    Returns:
    - Net contract value (float)
    """
    
    # Convert dates to pandas Timestamps
    inj_dates = pd.to_datetime(injection_dates)
    wdr_dates = pd.to_datetime(withdrawal_dates)
    
    # Track storage balance
    balance = 0
    total_costs = 0
    total_revenue = 0
    
    # Injection cash flows
    for d, vol in zip(inj_dates, injection_volumes):
        if balance + vol > max_storage_capacity:
            raise ValueError("Storage capacity exceeded!")
        price = estimate_price_func(d)
        balance += vol
        total_costs += price * vol  # purchase cost
        total_costs += injection_rate * vol  # injection fee
        total_costs += transport_fee_per_move  # transport fee
    
    # Withdrawal cash flows
    for d, vol in zip(wdr_dates, withdrawal_volumes):
        if balance - vol < 0:
            raise ValueError("Not enough inventory to withdraw!")
        price = estimate_price_func(d)
        balance -= vol
        total_revenue += price * vol  # sale revenue
        total_costs += withdrawal_rate * vol  # withdrawal fee
        total_costs += transport_fee_per_move  # transport fee
    
    # Storage fees: number of months between first injection and last withdrawal
    start = min(inj_dates)
    end = max(wdr_dates)
    months = (end.year - start.year) * 12 + (end.month - start.month) + 1
    total_costs += months * storage_fee_per_month
    
    # Net value
    net_value = total_revenue - total_costs
    return net_value

In [ ]:
# Example Usage

In [ ]:
# Suppose client injects in July 2023 and withdraws in January 2024
injection_dates = ["2023-07-31"]
withdrawal_dates = ["2024-01-31"]
injection_volumes = [1_000_000]   # 1 million MMBtu
withdrawal_volumes = [1_000_000]

# Costs
injection_rate = 0.01   # $0.01 per MMBtu
withdrawal_rate = 0.01  # $0.01 per MMBtu
max_storage_capacity = 2_000_000
storage_fee_per_month = 100_000
transport_fee_per_move = 50_000

# Using the estimate_price function from Task 1
net_val = price_storage_contract(
    injection_dates, withdrawal_dates,
    injection_volumes, withdrawal_volumes,
    injection_rate, withdrawal_rate,
    max_storage_capacity,
    storage_fee_per_month,
    transport_fee_per_move,
    estimate_price
)

print("Contract value: $", round(net_val, 2))